# 2. Остаточная тематическая модель (RTM)

Модель описана в разделе 3 статьи: кодировщик мешка слов, общая книга кодов тем, жадный остаточный подбор
(matching pursuit) и словарный декодер смеси тем поверх частотного фона. Обучение — алгоритм 2: Adam
(кодировщик и книга кодов с шагом 1e-3, логиты тем 3e-2), батч 256, nested dropout k_d ~ U{2..k_max},
перезапуск неиспользуемых тем каждые 5 эпох; 400 эпох на синтетике и 450 на реальных корпусах.
При одном и том же сиде и устройстве обучение повторяется до последнего знака.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import torch
from rtm.data import DATA, load_bow, background
from rtm.train import train_rtm, load_rtm

CORPUS, T, SEED = "lenta_E2", 10, 0
EPOCHS = 450           # как в статье; для пробного запуска можно взять 20
OUT = ROOT / "models" / "rtm" / CORPUS / f"s{SEED}" / f"T{T}"
model, log = train_rtm(DATA / CORPUS, T, SEED, OUT, epochs=EPOCHS)
log.tail()

{"epoch": 1, "loss": 9.128116214976592, "ppl": 3865.9274475773, "k_mean": 5.994335174560547, "restarted": 0, "sec": 0.8}


{"epoch": 51, "loss": 7.587116283528945, "ppl": 1970.802197581059, "k_mean": 6.03132963180542, "restarted": 0, "sec": 36.3}


{"epoch": 101, "loss": 7.553874646916109, "ppl": 1903.6557567858508, "k_mean": 6.0573410987854, "restarted": 0, "sec": 90.5}


{"epoch": 151, "loss": 7.555466161054723, "ppl": 1902.2534777183928, "k_mean": 6.028554916381836, "restarted": 0, "sec": 155.1}


{"epoch": 201, "loss": 7.557774599860696, "ppl": 1901.576980798426, "k_mean": 5.998381614685059, "restarted": 0, "sec": 220.1}


{"epoch": 251, "loss": 7.564075848635505, "ppl": 1898.6057489462698, "k_mean": 6.023815155029297, "restarted": 0, "sec": 283.7}


{"epoch": 301, "loss": 7.591328859329224, "ppl": 1925.3778530968245, "k_mean": 6.003352642059326, "restarted": 0, "sec": 347.6}


{"epoch": 351, "loss": 7.5577466628130745, "ppl": 1896.1682342216388, "k_mean": 5.98462438583374, "restarted": 0, "sec": 413.6}


{"epoch": 401, "loss": 7.6371996262494255, "ppl": 1946.0155569770088, "k_mean": 5.948670387268066, "restarted": 0, "sec": 477.8}


{"epoch": 450, "loss": 7.573950276655309, "ppl": 1908.0585168723228, "k_mean": 6.004046440124512, "restarted": 0, "sec": 544.9}


,epoch,loss,ppl,k_mean,restarted,sec
445,446,7.591360,1918.871258,5.973757,0,539.4
446,447,7.563200,1898.582770,5.973179,0,540.7
447,448,7.559886,1898.270302,6.012717,0,542.1
448,449,7.560882,1898.816915,5.953989,0,543.5
449,450,7.573950,1908.058517,6.004046,0,544.9


## Темы

Тема RTM — распределение φ_t = softmax(β_t + b) поверх фона; собственная часть softmax(β_t) совпадает с нормированным
лифтом φ_t / p_bg. Топ-слова выбираются по релевантности 0.6·ln φ + 0.4·ln(φ / p_bg), как в силуэте.

In [3]:
X, vocab = load_bow(DATA / CORPUS)
p_bg = background(X)
phi = np.load(OUT / "phi.npy")
rel = 0.6 * np.log(phi) + 0.4 * (np.log(phi) - np.log(p_bg))
for t in range(T):
    print(t, " ".join(vocab[w] for w in np.argsort(-rel[t])[:10]))

0

 полиция уголовный задерживать происходить сотрудник следственный орган пострадать правоохранительный мужчина
1 олимпийский спортсмен допинг спорт антидопинговый олимпиада соревнование игра чемпион проба
2 президент россия украина страна путин глава партия депутат владимир госдума
3 военный террористический группировка сирия исламский сила боевик запрещать государство россия
4 учёный исследование журнал университет исследователь земля обнаруживать клетка показывать позволять
5 пользователь сеть фильм видео актёр youtube ролик фотография страница соцсеть
6 самолёт борт турист авиакомпания рейс аэропорт пассажир путешественник лайнер экипаж
7 компания бренд коллекция релиз модель редакция основывать одежда выпускать представлять
8 процент рубль банк доллар нефть миллиард цена россия рынок компания
9 матч чемпионат команда сборный футбольный турнир лига сборная футбол тренер


## Темы документа

Для нового документа темы находятся одним проходом кодировщика и k_max шагами подбора: на выходе упорядоченная
последовательность тем и их веса θ (первая тема — главная).

In [4]:
m = load_rtm(OUT, X.shape[1])
x = torch.from_numpy(X[:5].toarray()).float()
chosen, theta = m.transform(x, k=4)
for i in range(5):
    print(i, [(t, round(float(v), 3)) for t, v in zip(chosen[i].tolist(), theta[i].tolist())])

0 [(3, 0.585), (4, 0.257), (2, 0.115), (8, 0.043)]
1 [(8, 0.763), (2, 0.219), (9, 0.012), (7, 0.006)]
2 [(7, 0.804), (5, 0.148), (2, 0.036), (8, 0.012)]
3 [(0, 0.837), (2, 0.089), (9, 0.047), (8, 0.026)]
4 [(7, 0.932), (8, 0.036), (2, 0.021), (9, 0.011)]


## Перебор числа тем

В статье RTM обучена на сетке `rtm.data.grid(корпус)` (T = 4…30 с шагом 2 и с шагом 1 в окне K ± 2) с тремя
сидами: около 600 моделей, 4–30 минут каждая на GPU. Ячейка ниже запускает весь перебор (модели, которые уже есть,
пропускаются).

In [5]:
from rtm.data import CORPORA, SEEDS, grid, epochs_rtm

RUN_SCAN = False
if RUN_SCAN:
    for c in CORPORA:
        for T_ in grid(c, "RTM"):
            for s in SEEDS:
                d = ROOT / "models" / "rtm" / c / f"s{s}" / f"T{T_}"
                if not (d / "phi.npy").exists():
                    train_rtm(DATA / c, T_, s, d, epochs=epochs_rtm(c), verbose=False)